In [ ]:
# =============================================================
# PHASTPADDING / ARFP  — FAITHFUL implementation (Chang et al. 2025, IEEE TNSRE,
#   "Forecast Padding Enhances Accuracy and Robustness of EEG-Phase-Synchronized TMS")
#   on Evoked Theta.   *** Corrected version of PHASTPADDING_theta_eval. ***
# =============================================================
# Only ONE thing changed vs the original PHASTPADDING notebook: the forecast-padding now
# follows the paper's step 3 -- the predicted (bandpass) signal is shifted to the raw EEG's
# LOCAL baseline before being appended ("noise padding"), instead of appending a zero-mean
# forecast and doing a single global de-mean. Without that shift the join discontinuity at
# t=0 scales with the window's DC/drift, and since the phase is read AT t=0 the resulting
# ringing corrupts it (the cause of the inflated MACE / near-zero PLV in the original run).
#
# Pipeline (Sec. II.A and the AR/ARFP steps in Sec. II.E.1):
#   1. signal = bandpass(raw window);  noise = raw window - signal
#   2. AR-forecast the SIGNAL forward (edge-trim -> aryule -> forward-predict)
#   3. padding = signal_forecast + LOCAL noise baseline (DC);  append to the RAW window
#   4. bandpass again -> Hilbert -> phase at t=0
#
# EVERYTHING ELSE IS UNCHANGED: data, GT (FIR-175 4-8 Hz filtfilt+Hilbert over the full
# epoch, read at the eval sample), channel, test split, metrics, JSON layout. So MACE/PLV
# here are directly comparable to the original PHASTPADDING run and to BOB/EPN/PEAP/PHASTIMATE
# (join on 'name'). Output goes to a SEPARATE folder/file -> does NOT overwrite the original.
#
# Tunable: BASE_MS (local-baseline window, ~one theta cycle). Params: phastimate-style
# {AR order 30, edge 20}, N_FORECAST = (FIR taps-1)/2 = 87 samp (covers the FIR-175 reach).

import numpy as np
import os, time, json, gc, glob, warnings
from scipy.signal import firwin, filtfilt, hilbert, welch
from scipy.stats import pearsonr, spearmanr
warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

STIM_DIR = '/content/drive/MyDrive/RAIN_IRC/data splits/onset'
OUTDIR   = '/content/drive/MyDrive/RAIN_IRC/results/PHASTPADDING_FAITHFUL_onset'   # separate dir
os.makedirs(OUTDIR, exist_ok=True)

files = sorted(glob.glob(os.path.join(STIM_DIR, '*.npz')))
assert files, f"no npz found in {STIM_DIR}"
_d0 = np.load(files[0], allow_pickle=True)
FS = float(_d0['fs']); PRE_SAMP = int(_d0['onset_in_epoch']); _d0.close()

EVAL_TIMES_S    = [0.3, 0.4, 0.5, 0.6, 0.7]
EVAL_TIMES_SAMP = [PRE_SAMP + int(round(t * FS)) for t in EVAL_TIMES_S]
THETA_SEARCH_LO, THETA_SEARCH_HI = 4.0, 8.0

# --- GT == forecast-padding's acausal filter: 4-8 Hz zero-phase FIR-175 (filtfilt) --- (UNCHANGED)
GT_BAND, GT_FIR_NUMTAPS = [4.0, 8.0], 175
gt_fir_b = firwin(GT_FIR_NUMTAPS, GT_BAND, fs=FS, window='hamming', pass_zero=False)

# --- forecast-padding params (paper, scaled 1kHz -> 300 Hz / theta) ---
INPUT_WIN  = int(round(0.980 * FS))   # 980 ms input length            -> 294
EDGE       = 20                       # phastimate edge 65@1kHz (65 ms) -> ~20 @300Hz
AR_ORDER   = 30                       # phastimate standard AR order
N_FORECAST = (GT_FIR_NUMTAPS - 1) // 2   # = 87 samp (~290 ms), covers the FIR-175 acausal reach
BASE_MS    = 120                      # local-baseline window for the noise padding (~one theta cycle)

print(f"\n{'='*94}")
print("PHASTPADDING / ARFP — FAITHFUL (Chang et al. 2025, IEEE TNSRE) on Evoked Theta — 300 Hz, FIR-175 GT")
print(f"{'='*94}")
print(f"  FS={FS:.0f} Hz | t=0 at sample {PRE_SAMP}")
print(f"  forecast = phastimate-style AR (order {AR_ORDER}, edge {EDGE}) on the FILTERED signal -> {N_FORECAST} samp "
      f"(~{N_FORECAST/FS*1000:.0f} ms; covers the FIR-175 reach)")
print(f"  pipeline: signal forecast + LOCAL noise baseline (BASE_MS={BASE_MS} ms) -> append to RAW {INPUT_WIN}-samp "
      f"window -> filtfilt(FIR-175 4-8) -> Hilbert @ t=0")
print(f"  *** baseline-shift / noise-padding is the fix vs the original; GT + everything else unchanged ***")
print(f"  Data: {STIM_DIR}  ({len(files)} subjects) | Output: {OUTDIR}")


# =============================================================
# forecast-padding internals
# =============================================================

def aryule(x, order):
    """MATLAB aryule: Levinson-Durbin on biased autocorrelation. a=[1,a1,...,ap]."""
    x = np.asarray(x, np.float64); n = len(x)
    r = np.correlate(x, x, 'full')[n - 1:n + order] / n
    a = np.zeros(order + 1); a[0] = 1.0; e = r[0]
    if e <= 1e-12:
        return a
    for m in range(1, order + 1):
        acc = r[m] + (np.dot(a[1:m], r[1:m][::-1]) if m > 1 else 0.0)
        k = -acc / e
        ao = a[1:m].copy(); a[1:m] = ao + k * ao[::-1]; a[m] = k
        e *= (1.0 - k * k)
        if e <= 1e-12:
            break
    return a

def _ff(b, x):
    return filtfilt(b, [1.0], x, padlen=min(3 * len(b), len(x) - 1))

def phastpad_predict(raw_1d, t0, input_window, fir_b, edge, ord, n_forecast, base_ms=None):
    """FAITHFUL forecast padding / ARFP (Chang et al. 2025). Forecast the FILTERED signal,
    shift it to the raw window's LOCAL baseline (noise padding), append to the RAW window,
    bandpass again, Hilbert, read phase at t=0. The baseline shift is the fix vs the original:
    it removes the DC/drift-driven discontinuity at the t=0 read point."""
    if base_ms is None:
        base_ms = BASE_MS
    start = t0 - input_window + 1
    if start < 0:
        return np.nan
    window = raw_1d[start:t0 + 1].astype(np.float64)        # RAW window, keep its baseline
    try:
        sig = _ff(fir_b, window)                            # signal = bandpassed rhythm
    except Exception:
        return np.nan
    sige = sig[edge:len(sig) - edge]
    if len(sige) < ord + 5 or np.std(sige) < 1e-10:
        return np.nan
    a = aryule(sige, ord); coeffs = -np.flip(a[1:]); L = len(sige)
    ext = np.empty(L + edge + n_forecast); ext[:L] = sige
    for k in range(edge + n_forecast):                      # forecast the SIGNAL (filtered domain)
        p = L + k; ext[p] = np.dot(coeffs, ext[p - ord:p])
    if not np.all(np.isfinite(ext)):
        return np.nan
    cap = 6.0 * np.std(sige)
    fc = np.clip(ext[L + edge:L + edge + n_forecast], -cap, cap)   # rhythm forecast for t>0
    nb = max(1, int(round(base_ms / 1000.0 * FS)))
    noise = window - sig
    baseline = float(np.mean(noise[-nb:]))                  # local pre-t0 baseline -> absorbs DC & drift
    padding = fc + baseline                                 # signal forecast shifted to raw baseline
    padded = np.concatenate([window, padding])              # smooth join (no zero-mean discontinuity)
    try:
        pf = _ff(fir_b, padded)                             # second convolution (bandpass removes DC itself)
    except Exception:
        return np.nan
    idx = input_window - 1                                  # phase at t=0
    if idx >= len(pf):
        return np.nan
    return float(np.angle(hilbert(pf)[idx]))


# =============================================================
# GT + metrics  (identical to BOB/EPN/PEAP/PHASTIMATE)   -- UNCHANGED
# =============================================================

def gt_phase_fir(epoch):
    try:
        return np.angle(hilbert(_ff(gt_fir_b, epoch.astype(np.float64))))
    except Exception:
        return np.full(len(epoch), np.nan)

def circ(a, b):
    return np.angle(np.exp(1j * (a - b)))

def errors_to_metrics(errors):
    errors = np.asarray(errors, float); errors = errors[np.isfinite(errors)]
    if len(errors) < 10:
        return {'mace': 90.0, 'plv': 0.0, 'csd': 180.0, 'acc': 0.0, 'bias': 0.0, 'n_samples': 0}
    ma = float(np.mean(np.abs(errors))); R = float(np.abs(np.mean(np.exp(1j * errors))))
    return {'mace': float(np.degrees(ma)), 'plv': R,
            'csd': float(np.degrees(np.sqrt(-2.0 * np.log(max(R, 1e-12))))),
            'acc': float((1.0 - ma / np.pi) * 100.0),
            'bias': float(np.degrees(np.angle(np.mean(np.exp(1j * errors))))),
            'n_samples': int(len(errors))}

def itpc_at(ph_list, trials, esamp):
    vals = [ph_list[tr][esamp] for tr in trials
            if esamp < len(ph_list[tr]) and np.isfinite(ph_list[tr][esamp])]
    return float(np.abs(np.mean(np.exp(1j * np.array(vals))))) if len(vals) >= 2 else float('nan')

def estimate_theta_peak(raw_1ch, onset_samp):
    seg = raw_1ch[:, onset_samp:]
    if seg.shape[1] < 16:
        return np.nan
    nfft = int(2 ** np.ceil(np.log2(seg.shape[1] * 4))); psd = None
    for tr in range(seg.shape[0]):
        f, px = welch(seg[tr].astype(np.float64), fs=FS, nperseg=seg.shape[1], nfft=nfft, detrend='linear')
        psd = px if psd is None else psd + px
    psd /= seg.shape[0]; band = (f >= THETA_SEARCH_LO) & (f <= THETA_SEARCH_HI)
    return float(f[band][np.argmax(psd[band])]) if np.any(band) else np.nan

def run_per_tp(raw, ph_fir, trials, epoch_len):
    per_tp = {t: [] for t in EVAL_TIMES_S}; n_eval = n_fail = 0
    for tr in trials:
        for t, es in zip(EVAL_TIMES_S, EVAL_TIMES_SAMP):
            if es >= epoch_len or es - INPUT_WIN + 1 < 0:
                continue
            n_eval += 1
            est = phastpad_predict(raw[tr], es, INPUT_WIN, gt_fir_b, EDGE, AR_ORDER, N_FORECAST)
            gf = ph_fir[tr][es]
            if not (np.isfinite(est) and np.isfinite(gf)):
                n_fail += 1; continue
            per_tp[t].append(circ(est, gf))
    return {t: np.array(v) for t, v in per_tp.items()}, (100.0 * n_fail / max(n_eval, 1))


Mounted at /content/drive

PHASTPADDING / ARFP — FAITHFUL (Chang et al. 2025, IEEE TNSRE) on Evoked Theta — 300 Hz, FIR-175 GT
  FS=300 Hz | t=0 at sample 300
  forecast = phastimate-style AR (order 30, edge 20) on the FILTERED signal -> 87 samp (~290 ms; covers the FIR-175 reach)
  pipeline: signal forecast + LOCAL noise baseline (BASE_MS=120 ms) -> append to RAW 294-samp window -> filtfilt(FIR-175 4-8) -> Hilbert @ t=0
  *** baseline-shift / noise-padding is the fix vs the original; GT + everything else unchanged ***
  Data: /content/drive/MyDrive/RAIN_IRC/data splits/onset  (20 subjects) | Output: /content/drive/MyDrive/RAIN_IRC/results/PHASTPADDING_FAITHFUL_onset


In [ ]:
# =============================================================
# Main loop  [forecast-padding fixed pipeline; scored on the npz TEST split, per timepoint]
# =============================================================
print(f"\n{'Subj':<14s}{'N':>5s}{'MACE':>7s}{'PLV':>7s}{'CSD':>7s}{'Acc':>7s}{'bias':>7s}{'pk':>5s}{'SNR':>6s}{'ITPC':>6s}{'fail%':>6s}{'t':>5s}")
print("-" * 96)

results = []
for fp in files:
    t0 = time.time(); d = np.load(fp, allow_pickle=True); name = os.path.basename(fp)[:-4]
    if float(d['fs']) != FS or int(d['onset_in_epoch']) != PRE_SAMP:
        print(f"{name}: geometry mismatch, skipping"); d.close(); continue
    epochs = d['epochs']; best = int(d['best_ch_idx']); ch_name = str(d['best_ch_name'])
    split_arr = np.array([str(x) for x in d['split']]); theta_snr = float(d['ch_theta_snr'][best])
    raw = epochs[:, best, :].astype(np.float64)
    n_trials, n_ch, epoch_len = epochs.shape
    test_trials = np.where(split_arr == 'test')[0]

    ph_fir = [gt_phase_fir(raw[tr]) for tr in range(n_trials)]
    itpc_tp = {t: (itpc_at(ph_fir, test_trials, es) if es < epoch_len else float('nan'))
               for t, es in zip(EVAL_TIMES_S, EVAL_TIMES_SAMP)}
    itpc_mean = float(np.nanmean(list(itpc_tp.values())))
    f_peak = estimate_theta_peak(raw, PRE_SAMP)

    tp, fail = run_per_tp(raw, ph_fir, test_trials, epoch_len)
    pool = (np.concatenate([tp[t] for t in EVAL_TIMES_S])
            if any(len(tp[t]) for t in EVAL_TIMES_S) else np.array([]))
    m = errors_to_metrics(pool); tp_m = {t: errors_to_metrics(tp[t]) for t in EVAL_TIMES_S}

    dt = time.time() - t0; pk = f"{f_peak:.1f}" if np.isfinite(f_peak) else "n/a"
    print(f"{name:<14s}{m['n_samples']:>5d}{m['mace']:>6.1f}°{m['plv']:>7.3f}{m['csd']:>6.1f}°"
          f"{m['acc']:>6.1f}%{m['bias']:>+6.1f}°{pk:>5s}{theta_snr:>6.2f}{itpc_mean:>6.3f}{fail:>5.1f}%{dt:>4.0f}s")

    results.append({
        'name': name, 'best_ch': ch_name, 'n_trials': int(n_trials),
        'n_test_samples': m['n_samples'], 'fail_pct': fail, 'theta_snr': theta_snr,
        'theta_itpc_per_timepoint': {str(t): itpc_tp[t] for t in EVAL_TIMES_S},
        'theta_itpc_mean': itpc_mean,
        'theta_peak_hz': float(f_peak) if np.isfinite(f_peak) else None,
        'metrics_pooled': m,
        'metrics_per_timepoint': {str(t): tp_m[t] for t in EVAL_TIMES_S},
        'errors_per_timepoint_rad': {str(t): [float(x) for x in tp[t]] for t in EVAL_TIMES_S}})
    d.close(); del epochs, raw, ph_fir; gc.collect()

print(f"\n{len(results)} subjects done")



Subj              N   MACE    PLV    CSD    Acc   bias   pk   SNR  ITPC fail%    t
------------------------------------------------------------------------------------------------
Subject10001    250  56.9°  0.460  71.4°  68.4%  +6.5°  4.0  2.56 0.097  0.0%   2s
Subject1001     250  48.9°  0.551  62.5°  72.8%  +4.1°  4.0  2.59 0.162  0.0%   3s
Subject11002    250  57.0°  0.429  74.5°  68.3%  -5.4°  4.0  2.04 0.128  0.0%   3s
Subject12001    250  51.0°  0.523  65.2°  71.7%  +1.7°  4.0  1.93 0.179  0.0%   3s
Subject13001    250  42.1°  0.637  54.4°  76.6%  +3.6°  6.9  2.65 0.072  0.0%   4s
Subject14001    250  69.8°  0.283  91.0°  61.2%  +5.8°  4.0  2.41 0.169  0.0%   3s
Subject15001    250  59.6°  0.414  76.1°  66.9%  -0.1°  4.0  2.36 0.146  0.0%   3s
Subject16001    250  50.8°  0.526  64.9°  71.8%  +6.0°  4.0  1.74 0.192  0.0%   3s
Subject17001    250  53.5°  0.489  68.5°  70.3%  +4.7°  4.0  2.45 0.205  0.0%   3s
Subject18002    250  57.3°  0.476  69.8°  68.1% -22.0°  7.8  2.99 0.121 

In [ ]:
# =============================================================
# SUMMARY + SAVE  (model-tagged JSON, PEAP-parallel: single fixed condition)
# =============================================================
valid = [r for r in results if r['n_test_samples'] >= 10]; n = len(valid)
M = np.array([r['metrics_pooled']['mace'] for r in valid])
P = np.array([r['metrics_pooled']['plv'] for r in valid])
C = np.array([r['metrics_pooled']['csd'] for r in valid])
A = np.array([r['metrics_pooled']['acc'] for r in valid])
B = np.array([r['metrics_pooled']['bias'] for r in valid])
SNR = np.array([r['theta_snr'] for r in valid]); IT = np.array([r['theta_itpc_mean'] for r in valid])

print('=' * 72); print('PHASTPADDING (FAITHFUL) THETA — fixed pipeline (single condition)'); print('=' * 72)
print(f"  MACE {M.mean():.1f}°±{M.std():.1f}   PLV {P.mean():.3f}   CSD {C.mean():.1f}°   Acc {A.mean():.1f}%")
print(f"  bias {B.mean():+.1f}° (mean) | per-subject |bias| mean {np.mean(np.abs(B)):.1f}°  <- with the baseline-shift fix, errors should be coherent (PLV up); residual bias = paper's bandpass shift")
print(f"  mean failure rate {np.mean([r['fail_pct'] for r in results]):.1f}%")
print(f"  (n={n} unique participants; RAW errors, no bias correction)")

print('\n--- Per-timepoint ---')
print(f"{'ms':>5}{'MACE':>8}{'PLV':>8}{'CSD':>8}{'Acc':>8}{'bias':>8}{'ITPC':>8}{'N':>7}")
for t in EVAL_TIMES_S:
    g = lambda k: np.array([r['metrics_per_timepoint'][str(t)][k] for r in valid])
    ti = np.array([r['theta_itpc_per_timepoint'][str(t)] for r in valid])
    print(f"{int(t*1000):>5}{g('mace').mean():>7.1f}°{g('plv').mean():>8.3f}{g('csd').mean():>7.1f}°"
          f"{g('acc').mean():>7.1f}%{g('bias').mean():>+7.1f}°{np.nanmean(ti):>8.3f}{g('n_samples').mean():>7.0f}")

group_stats = {'n_participants': int(n), 'mean_mace': float(M.mean()), 'mean_plv': float(P.mean()),
               'mean_csd': float(C.mean()), 'mean_acc': float(A.mean()), 'mean_bias': float(B.mean()),
               'mean_abs_bias': float(np.mean(np.abs(B))),
               'mean_fail_pct': float(np.mean([r['fail_pct'] for r in results]))}

print('\n--- Accuracy correlations across subjects ---')
if n >= 3 and np.std(SNR) > 0:
    r1, p1 = pearsonr(SNR, A); rs1, ps1 = spearmanr(SNR, A)
    group_stats.update({'acc_vs_snr_pearson_r': float(r1), 'acc_vs_snr_pearson_p': float(p1),
                        'acc_vs_snr_spearman_r': float(rs1), 'acc_vs_snr_spearman_p': float(ps1)})
    print(f"  Acc vs SNR : r={r1:+.3f} (p={p1:.4f})  rho={rs1:+.3f} (p={ps1:.4f})")
if n >= 3 and np.std(IT) > 0:
    r2, p2 = pearsonr(IT, A); rs2, ps2 = spearmanr(IT, A)
    group_stats.update({'acc_vs_itpc_pearson_r': float(r2), 'acc_vs_itpc_pearson_p': float(p2),
                        'acc_vs_itpc_spearman_r': float(rs2), 'acc_vs_itpc_spearman_p': float(ps2)})
    print(f"  Acc vs ITPC: r={r2:+.3f} (p={p2:.4f})  rho={rs2:+.3f} (p={ps2:.4f})")

out = {'model': 'PHASTPADDING_FAITHFUL',
       'config': {'fs': FS, 'pre_samp': PRE_SAMP, 'eval_times_s': EVAL_TIMES_S, 'horizon_ms': 0.0,
                  'gt_band': GT_BAND, 'gt_fir_numtaps': GT_FIR_NUMTAPS,
                  'input_window': INPUT_WIN, 'edge': EDGE, 'ar_order': AR_ORDER, 'n_forecast': N_FORECAST,
                  'base_ms': BASE_MS,
                  'method': 'Chang et al. 2025 (IEEE TNSRE) forecast padding / ARFP: bandpass -> AR-forecast the '
                            'signal -> shift forecast to local raw baseline (noise padding) -> append to raw window '
                            '-> bandpass again (FIR-175) -> Hilbert @ t=0',
                  'forecast_engine': 'filter -> edge-trim -> aryule (Yule-Walker) -> forward-predict the filtered signal',
                  'noise_padding': 'predicted signal shifted to local raw baseline (DC of raw-signal over last BASE_MS ms)',
                  'acausal_filter': 'FIR-175 4-8 Hz Hamming filtfilt (== GT filter)',
                  'data_dir': STIM_DIR, 'split': 'chronological cal/test from npz split array',
                  'channel': 'best_ch_idx from npz', 'conditions': ['single (fixed pipeline)'],
                  'itpc_basis': 'GT-phase ITPC, test trials, per eval point', 'errors_unit': 'radians',
                  'bias_note': 'RAW errors (no correction). Baseline-shift fix applied -> phase read at t=0 sits off '
                               'the join discontinuity; any residual systematic `bias` is the paper bandpass shift'},
       'group_stats': group_stats, 'subjects': results}
outpath = os.path.join(OUTDIR, 'phastpadding_faithful_theta_results.json')
with open(outpath, 'w') as f:
    json.dump(out, f, indent=2, default=str)
print(f"\nSaved -> {outpath}")
print("Per subject: metrics_pooled (incl. bias) + per-timepoint + errors (rad) + SNR/ITPC/peak + fail_pct.")
print("Same data/GT/channel/test-split as BOB/EPN/PEAP/PHASTIMATE -> join on 'name'.")
print("Compare PHASTPADDING_FAITHFUL.metrics_pooled <-> the original PHASTPADDING.metrics_pooled and <-> PEAP.")


PHASTPADDING (FAITHFUL) THETA — fixed pipeline (single condition)
  MACE 54.4°±7.4   PLV 0.482   CSD 69.6°   Acc 69.8%
  bias +0.9° (mean) | per-subject |bias| mean 5.2°  <- with the baseline-shift fix, errors should be coherent (PLV up); residual bias = paper's bandpass shift
  mean failure rate 0.0%
  (n=20 unique participants; RAW errors, no bias correction)

--- Per-timepoint ---
   ms    MACE     PLV     CSD     Acc    bias    ITPC      N
  300   54.5°   0.489   69.0°   69.7%   +3.8°   0.121     50
  400   55.1°   0.477   70.4°   69.4%   -1.5°   0.130     50
  500   56.2°   0.476   70.4°   68.8%   +1.7°   0.157     50
  600   51.9°   0.522   65.8°   71.2%   -1.8°   0.157     50
  700   54.6°   0.486   69.3°   69.7%   +3.3°   0.138     50

--- Accuracy correlations across subjects ---
  Acc vs SNR : r=+0.036 (p=0.8815)  rho=+0.233 (p=0.3227)
  Acc vs ITPC: r=+0.180 (p=0.4479)  rho=+0.138 (p=0.5608)

Saved -> /content/drive/MyDrive/RAIN_IRC/results/PHASTPADDING_FAITHFUL_onset/phastp